# 2. Caption

This notebook extracts frozen CNN features for Flickr8k and prepares an image-level train, validation, and test split. Vocabulary is built from training captions only.

In [ ]:
from pathlib import Path
import sys

def find_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start] + list(start.parents):
        if (path / 'src').exists() and (path / 'data').exists():
            return path
    raise RuntimeError('Repo root tidak ditemukan. Jalankan dari repo, atau run Setup lebih dulu.')

ROOT = find_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('ROOT:', ROOT)


In [ ]:
import json
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from cnn.utility import image_paths, feature_extractor
from rnn.preprocess import clean_text, tokenize, make_vocab, pad_seq

SEED = 42
MAX_SEQUENCE_LENGTH = 34
MIN_WORD_FREQ = 1
FEATURE_BATCH = 64
IMAGE_SIZE = (299, 299)
TRAIN_RATIO = 0.75
VAL_RATIO = 0.125
USE_KAGGLE_RNN_ARTIFACTS = True
REBUILD_PROCESSED = False

RAW_DIR = ROOT / 'data/raw/flickr8k'
IMAGE_DIR = RAW_DIR / 'Images'
CAPTION_PATH = RAW_DIR / 'captions.txt'
FEATURE_DIR = ROOT / 'data/features'
PROC_DIR = ROOT / 'data/processed/flickr8k'
TABLE_DIR = ROOT / 'reports/tables'
for folder in [FEATURE_DIR, PROC_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


In [ ]:
def parse_captions(path):
    by_image = defaultdict(list)
    with Path(path).open('r', encoding='utf-8') as file:
        for raw in file:
            line = raw.strip()
            if not line:
                continue
            if ',' in line:
                image_raw, caption = line.split(',', 1)
            elif '	' in line:
                image_raw, caption = line.split('	', 1)
            else:
                continue
            image_id = Path(image_raw.split('#')[0]).name
            if image_id.lower() in {'image', 'filename'}:
                continue
            caption = clean_text(caption)
            if caption:
                by_image[image_id].append(caption)
    return dict(by_image)

caption_by_image = parse_captions(CAPTION_PATH)
print('caption images:', len(caption_by_image))
print('captions:', sum(len(v) for v in caption_by_image.values()))


In [ ]:
feature_path = FEATURE_DIR / 'flickr8k_features.npy'
feature_id_path = FEATURE_DIR / 'flickr8k_image_ids.txt'
encoder_path = ROOT / 'models/cnn/flickr8k_inceptionv3_encoder.keras'
encoder_path.parent.mkdir(parents=True, exist_ok=True)

if feature_path.exists() and feature_id_path.exists():
    features = np.load(feature_path)
    feature_ids = [line.strip() for line in feature_id_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('skip feature extraction:', features.shape)
else:
    paths = image_paths(IMAGE_DIR)
    cached_weights = Path.home() / '.keras/models/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5'
    weights_arg = str(cached_weights) if cached_weights.exists() else 'imagenet'
    encoder = tf.keras.applications.InceptionV3(include_top=False, weights=weights_arg, pooling='avg')
    encoder.trainable = False
    encoder.save(encoder_path)
    features = feature_extractor(
        paths,
        encoder,
        feature_path,
        target_size=IMAGE_SIZE,
        batch_size=FEATURE_BATCH,
        image_id_path=feature_id_path,
        preprocess_fn=tf.keras.applications.inception_v3.preprocess_input,
    )
    feature_ids = [line.strip() for line in feature_id_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('saved:', features.shape)

# Older feature files may store stems. Normalize to filenames when possible.
if feature_ids and not feature_ids[0].lower().endswith(('.jpg', '.jpeg', '.png')):
    image_name_lookup = {Path(path).stem: Path(path).name for path in image_paths(IMAGE_DIR)}
    feature_ids = [image_name_lookup.get(image_id, image_id) for image_id in feature_ids]
    feature_id_path.write_text('\n'.join(feature_ids), encoding='utf-8')
    print('normalized feature ids to filenames')


In [ ]:
feature_ids = [line.strip() for line in feature_id_path.read_text(encoding='utf-8').splitlines() if line.strip()]
features = np.load(feature_path).astype('float32')
valid_ids = sorted(set(feature_ids).intersection(caption_by_image))
split_path = PROC_DIR / 'split.json'

if USE_KAGGLE_RNN_ARTIFACTS and split_path.exists():
    split = json.loads(split_path.read_text(encoding='utf-8'))
    print('using existing image-level split:', {key: len(value) for key, value in split.items()})
else:
    rng = np.random.default_rng(SEED)
    order = np.array(valid_ids, dtype=object)
    rng.shuffle(order)
    n_train = int(round(len(order) * TRAIN_RATIO))
    n_val = int(round(len(order) * VAL_RATIO))
    split = {
        'train': order[:n_train].tolist(),
        'val': order[n_train:n_train+n_val].tolist(),
        'test': order[n_train+n_val:].tolist(),
    }
    split_path.write_text(json.dumps(split, indent=2), encoding='utf-8')
    print('created split:', {key: len(value) for key, value in split.items()})


In [ ]:
required_arrays = [PROC_DIR / f'{name}_{kind}.npy' for name in ['train', 'val', 'test'] for kind in ['features', 'captions']]
required_ids = [PROC_DIR / f'{name}_image_ids.txt' for name in ['train', 'val', 'test']]
vocab_path = PROC_DIR / 'vocab.json'

if (not REBUILD_PROCESSED) and vocab_path.exists() and all(path.exists() for path in required_arrays + required_ids):
    word_to_index = json.loads(vocab_path.read_text(encoding='utf-8'))
    summary = {
        'max_sequence_length': MAX_SEQUENCE_LENGTH,
        'vocab_size': len(word_to_index),
        'split_images': {key: len(value) for key, value in split.items()},
        'source': 'existing_processed_arrays',
    }
    print('processed caption arrays already exist')
else:
    train_captions = [caption for image_id in split['train'] for caption in caption_by_image[image_id]]
    if vocab_path.exists() and USE_KAGGLE_RNN_ARTIFACTS:
        word_to_index = json.loads(vocab_path.read_text(encoding='utf-8'))
        print('using existing train vocabulary:', len(word_to_index))
    else:
        word_to_index, _ = make_vocab(train_captions, special=['<pad>', '<start>', '<end>', '<unk>'], min_freq=MIN_WORD_FREQ)
        vocab_path.write_text(json.dumps(word_to_index, indent=2), encoding='utf-8')

    feature_lookup = {image_id: features[index] for index, image_id in enumerate(feature_ids)}

    def encode_caption(caption):
        pad_id = word_to_index['<pad>']
        ids = [word_to_index['<start>']]
        ids.extend(word_to_index.get(token, word_to_index.get('<unk>', pad_id)) for token in tokenize(caption))
        ids.append(word_to_index['<end>'])
        return pad_seq(ids, MAX_SEQUENCE_LENGTH, pad_id=pad_id)

    def build_rows(image_ids):
        row_features = []
        row_sequences = []
        row_ids = []
        missing = []
        for image_id in image_ids:
            if image_id not in feature_lookup or image_id not in caption_by_image:
                missing.append(image_id)
                continue
            for caption in caption_by_image[image_id]:
                row_features.append(feature_lookup[image_id])
                row_sequences.append(encode_caption(caption))
                row_ids.append(image_id)
        if missing:
            print('missing image ids:', len(missing))
        return np.asarray(row_features, dtype='float32'), np.asarray(row_sequences, dtype='int32'), row_ids

    for name in ['train', 'val', 'test']:
        row_features, row_sequences, row_ids = build_rows(split[name])
        np.save(PROC_DIR / f'{name}_features.npy', row_features)
        np.save(PROC_DIR / f'{name}_captions.npy', row_sequences)
        (PROC_DIR / f'{name}_image_ids.txt').write_text('\n'.join(row_ids), encoding='utf-8')
        print(name, row_features.shape, row_sequences.shape)

    summary = {
        'max_sequence_length': MAX_SEQUENCE_LENGTH,
        'vocab_size': len(word_to_index),
        'split_images': {key: len(value) for key, value in split.items()},
        'source': 'rebuilt_from_split_and_features',
    }

(TABLE_DIR / 'caption_data_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary
